<a href="https://colab.research.google.com/github/Bebarzzz/machine-/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# The models
from sklearn.ensemble import RandomForestClassifier

In [12]:
df = pd.read_csv('Chronotype_NHANES_Imputation1.csv')
df.head()



,Seqn,Gender,Age,Race,BMI,Waist_C,Systolic,Diastolic,Carb_diet,HSCRP,Smokingstatus,Alcohol,Sleep_hrs,Sleeptime,Wakeuptime,Chronotype_slphrs,WakeUpCat
0,83732,1,62,2.0,27.8,101.1,122.6667,65.33334,126.0,0.6,3.0,1.0,5.5,23:30:00,05:00:00,3.0,1
1,83733,1,53,2.0,30.8,107.9,140.0000,86.00000,126.0,1.4,1.0,7.0,8.0,23:00:00,07:00:00,3.0,3
2,83734,1,78,2.0,28.8,116.5,135.3333,45.33333,96.0,0.6,3.0,0.0,7.0,22:30:00,05:30:00,2.0,2
3,83735,2,56,2.0,42.4,110.1,134.0000,70.00000,216.0,9.0,3.0,3.0,6.5,23:30:00,06:00:00,3.0,2
4,83741,1,22,3.0,28.0,86.6,111.3333,72.66666,5.5,1.3,2.0,3.0,6.5,23:00:00,05:30:00,3.0,2


### Data Preparation
We need to separate the target variable `Chronotype_slphrs` from the features. We should also remove identifiers like `Seqn` and non-numeric time columns that aren't ready for modeling.

In [13]:
# Define features and target
# Dropping Seqn (ID) and the original time strings which aren't numeric
X = df.drop(columns=['Chronotype_slphrs', 'Seqn', 'Sleeptime', 'Wakeuptime'])
y = df['Chronotype_slphrs']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (4469, 13)
Testing set shape: (1118, 13)


### Model Training and Evaluation
Now we initialize and fit the Random Forest Classifier.

In [14]:
# Initialize the revised model with balanced class weights and tuned depth
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42
)

# Fit the model
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Evaluation
print("Revised Random Forest Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Revised Random Forest Accuracy Score: 0.740608228980322

Classification Report:
              precision    recall  f1-score   support

         1.0       0.71      0.51      0.59       156
         2.0       0.70      0.76      0.73       344
         3.0       0.78      0.86      0.82       443
         4.0       0.67      0.62      0.64        74
         5.0       0.80      0.58      0.67       101

    accuracy                           0.74      1118
   macro avg       0.73      0.67      0.69      1118
weighted avg       0.74      0.74      0.74      1118



### XGBoost Classifier
Next, we'll try the XGBoost model. First, we need to import it and then train it using our prepared features and target.

In [17]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost often requires label encoding for targets starting from 0
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

# Initialize the XGBoost model (removed deprecated use_label_encoder)
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')

# Fit the model
xgb_model.fit(X_train, y_train_encoded)

# Make predictions
y_pred_xgb = xgb_model.predict(X_test)

# Evaluation
print("XGBoost Accuracy Score:", accuracy_score(y_test_encoded, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_test_encoded, y_pred_xgb, target_names=le.classes_.astype(str)))

XGBoost Accuracy Score: 0.7397137745974955

Classification Report:
              precision    recall  f1-score   support

         1.0       0.64      0.49      0.56       156
         2.0       0.70      0.70      0.70       344
         3.0       0.78      0.88      0.83       443
         4.0       0.77      0.68      0.72        74
         5.0       0.78      0.67      0.72       101

    accuracy                           0.74      1118
   macro avg       0.73      0.69      0.71      1118
weighted avg       0.74      0.74      0.73      1118



### Neural Network Hyperparameter Tuning with Keras Tuner

We will now use `keras_tuner` to perform a more extensive hyperparameter search for our Neural Network model. This allows us to systematically explore different architectures and training parameters to find the best performing model. The goal is to maximize the validation accuracy.

In [19]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'class_weight': ['balanced'] # Address class imbalance
}

# Initialize the SVM classifier
svm_tuned = SVC(random_state=42, decision_function_shape='ovr')

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=svm_tuned, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='f1_macro')

# Fit GridSearchCV on the scaled training data
grid_search.fit(X_train_scaled, y_train_encoded)

# Get the best parameters and best score
print("Best parameters found: ", grid_search.best_params_)
print("Best macro F1-score found: ", grid_search.best_score_)

# Make predictions with the best estimator
y_pred_tuned_svm = grid_search.best_estimator_.predict(X_test_scaled)

# Evaluate the tuned model
print("\nSVM Tuned Accuracy Score:", accuracy_score(y_test_encoded, y_pred_tuned_svm))
print("\nClassification Report for Tuned SVM:")
print(classification_report(y_test_encoded, y_pred_tuned_svm, target_names=le.classes_.astype(str)))

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best parameters found:  {'C': 10, 'class_weight': 'balanced', 'kernel': 'rbf'}
Best macro F1-score found:  0.6506775524564037

SVM Tuned Accuracy Score: 0.7012522361359571

Classification Report for Tuned SVM:
              precision    recall  f1-score   support

         1.0       0.54      0.60      0.57       156
         2.0       0.69      0.68      0.68       344
         3.0       0.82      0.79      0.80       443
         4.0       0.55      0.69      0.61        74
         5.0       0.64      0.56      0.60       101

    accuracy                           0.70      1118
   macro avg       0.65      0.66      0.65      1118
weighted avg       0.71      0.70      0.70      1118



In [32]:
# Install Keras Tuner if not already installed
import subprocess
import sys

try:
    import keras_tuner
except ImportError:
    print("Installing Keras Tuner...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "keras-tuner"])
    import keras_tuner

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

print("Keras Tuner and TensorFlow imported successfully.")

Keras Tuner and TensorFlow imported successfully.


In [33]:
# Scale features for Neural Networks
scaler = StandardScaler()
X_train_nn_scaled = scaler.fit_transform(X_train)
X_test_nn_scaled = scaler.transform(X_test)

# Get number of classes for the output layer
num_classes = len(np.unique(y_train_encoded))

# Define the Keras Tuner model building function
def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_nn_scaled.shape[1],)))

    # Tune the number of layers
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(layers.Dense(units=hp.Int('units_' + str(i),
                                            min_value=32,
                                            max_value=512,
                                            step=32),
                                 activation='relu'))
        model.add(layers.Dropout(hp.Float('dropout_' + str(i),
                                           min_value=0.0,
                                           max_value=0.5,
                                           step=0.1)))

    # Output layer
    model.add(layers.Dense(num_classes, activation='softmax'))

    # Tune the learning rate for the optimizer
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Initialize the RandomSearch tuner
tuner = keras_tuner.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,  # Number of different models to try
    executions_per_trial=2, # Number of models to train for each trial (average performance)
    directory='keras_tuner_dir', # Directory to store results
    project_name='chronotype_mlp_tuning')

# Display search space summary
tuner.search_space_summary()

# Perform the search
print("\nStarting hyperparameter search...")
tuner.search(X_train_nn_scaled, y_train_encoded,
             epochs=20, # Number of epochs for each model in a trial
             validation_data=(X_test_nn_scaled, y_test_encoded),
             callbacks=[tf.keras.callbacks.EarlyStopping('val_loss', patience=3)])

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"\nThe optimal number of layers is {best_hps.get('num_layers')}.")
for i in range(best_hps.get('num_layers')):
    print(f"The optimal number of units in layer {i} is {best_hps.get('units_' + str(i))}.")
    print(f"The optimal dropout rate in layer {i} is {best_hps.get('dropout_' + str(i))}.")
print(f"The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}.")

# Train the best model found by the tuner
best_model = tuner.get_best_models(num_models=1)[0]
history = best_model.fit(X_train_nn_scaled, y_train_encoded,
                         epochs=50, # Train for more epochs with the best model
                         validation_data=(X_test_nn_scaled, y_test_encoded),
                         callbacks=[tf.keras.callbacks.EarlyStopping('val_loss', patience=5)], # Early stopping for final training
                         verbose=1)

# Evaluate the best model
loss, accuracy = best_model.evaluate(X_test_nn_scaled, y_test_encoded, verbose=0)
print(f"\nBest Tuned Neural Network Accuracy: {accuracy:.4f}")

# Make predictions with the best model
y_pred_nn_tuned = np.argmax(best_model.predict(X_test_nn_scaled), axis=1)

# Classification Report
print("\nClassification Report for Best Tuned Neural Network:")
print(classification_report(y_test_encoded, y_pred_nn_tuned, target_names=le.classes_.astype(str)))

Trial 10 Complete [00h 00m 22s]
val_accuracy: 0.7450805306434631

Best val_accuracy So Far: 0.7669946253299713
Total elapsed time: 00h 05m 52s

The optimal number of layers is 3.
The optimal number of units in layer 0 is 480.
The optimal dropout rate in layer 0 is 0.0.
The optimal number of units in layer 1 is 32.
The optimal dropout rate in layer 1 is 0.0.
The optimal number of units in layer 2 is 32.
The optimal dropout rate in layer 2 is 0.0.
The optimal learning rate for the optimizer is 0.001.
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7722 - loss: 0.5790 - val_accuracy: 0.7630 - val_loss: 0.6080
Epoch 2/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7874 - loss: 0.5526 - val_accuracy: 0.7504 - val_loss: 0.5963
Epoch 3/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7908 - loss: 0.5417 - val_accuracy: 0.7504 - val_loss: 0.5964
Epoch 4/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7861 - loss: 0.5363 - val_accuracy: 0.7674 - val_loss: 0.5793
Epoch 5/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7912 - loss: 0.5206 - val_accuracy: 0.7648 - val_loss: 0.6159
Epoch 6/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7901 - loss: 0.5163 - val_accuracy: 0.7639 - val_loss: 0.5699
Epoch 7/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7917 - loss: 0.5049 - val_accuracy: 0.7639 - val_loss: 0.5913
Epoch 8/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7941 - loss: 0.5036 - val_accuracy: 0.7415 - val_